In [0]:
# Setup & imports
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import tempfile, os, json

spark = SparkSession.builder.getOrCreate()

In [0]:
# Create two prediction tables in my own database

YOUR_DB = "gr5069.sz3388"  

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {YOUR_DB}.hw5_rf_predictions (
        raceId        INT,
        driverId      INT,
        constructorId INT,
        grid          INT,
        actual_podium INT,
        predicted_podium INT,
        predicted_proba  DOUBLE,
        model_name    STRING,
        logged_at     TIMESTAMP
    )
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {YOUR_DB}.hw5_gb_predictions (
        raceId        INT,
        driverId      INT,
        constructorId INT,
        grid          INT,
        actual_podium INT,
        predicted_podium INT,
        predicted_proba  DOUBLE,
        model_name    STRING,
        logged_at     TIMESTAMP
    )
""")

print("✅ Tables created:")
print(f"  {YOUR_DB}.hw5_rf_predictions")
print(f"  {YOUR_DB}.hw5_gb_predictions")

In [0]:
# Load F1 data from Unity Catalog Volume
results_df   = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv",   header=True, inferSchema=True).toPandas()
races_df     = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv",     header=True, inferSchema=True).toPandas()
drivers_df   = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv",   header=True, inferSchema=True).toPandas()
constructors_df = spark.read.csv("/Volumes/gr5069/raw/f1_data/constructors.csv", header=True, inferSchema=True).toPandas()
pit_stops_df = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True, inferSchema=True).toPandas()

print("Shapes:", results_df.shape, races_df.shape)
results_df.head(3)

In [0]:
# Feature engineering — predict whether a driver finishes on the podium (top 3)
df = results_df.merge(races_df[['raceId', 'year', 'circuitId']], on='raceId', how='left')

# Target: podium finish (position 1, 2, or 3)
df['positionOrder'] = pd.to_numeric(df['positionOrder'], errors='coerce')
df['podium'] = (df['positionOrder'] <= 3).astype(int)

# Pit stop count per driver per race
pit_counts = pit_stops_df.groupby(['raceId', 'driverId']).size().reset_index(name='pit_stop_count')
df = df.merge(pit_counts, on=['raceId', 'driverId'], how='left')
df['pit_stop_count'] = df['pit_stop_count'].fillna(0)

# Historical avg finishing position per driver (prior races only — simple approximation)
df = df.sort_values(['driverId', 'year', 'raceId'])
df['driver_avg_pos'] = (
    df.groupby('driverId')['positionOrder']
      .transform(lambda x: x.shift(1).expanding().mean())
)

# Numeric coercion
for col in ['grid', 'constructorId', 'circuitId', 'year']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with NaN in key features
FEATURES = ['grid', 'constructorId', 'circuitId', 'year', 'pit_stop_count', 'driver_avg_pos']
TARGET   = 'podium'

df_model = df[FEATURES + [TARGET, 'raceId', 'driverId']].dropna()
print(f"Modeling dataset: {df_model.shape}")
print(f"Podium rate: {df_model[TARGET].mean():.3f}")

In [0]:
# Train/test split
X = df_model[FEATURES]
y = df_model[TARGET]
meta = df_model[['raceId', 'driverId', 'constructorId' if 'constructorId' in df_model.columns else 'constructorId']].copy()

# Re-attach constructorId to meta from original
meta = df_model[['raceId', 'driverId']].copy()
meta['constructorId'] = df['constructorId'].reindex(df_model.index).values
meta['grid'] = df_model['grid'].values

X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X, y, meta, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [0]:
# Helper — create artifacts
def save_confusion_matrix(y_true, y_pred, title, path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    plt.tight_layout()
    fig.savefig(path)
    plt.close(fig)

def save_feature_importance(model, feature_names, path):
    importances = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(6, 4))
    importances.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title("Feature Importances")
    plt.tight_layout()
    fig.savefig(path)
    plt.close(fig)

In [0]:
# Model 1 — Random Forest
RF_PARAMS = {
    "n_estimators":   200,
    "max_depth":       8,
    "min_samples_split": 10,
    "class_weight":  "balanced",
    "random_state":   42,
}

mlflow.set_experiment("/Users/sz3388@columbia.edu/hw5_f1_podium_prediction")

with mlflow.start_run(run_name="RandomForest_Podium") as rf_run:

    # --- Train ---
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_train, y_train)

    # --- Predict ---
    y_pred_rf  = rf.predict(X_test)
    y_proba_rf = rf.predict_proba(X_test)[:, 1]

    # --- 4 Metrics ---
    acc  = accuracy_score(y_test, y_pred_rf)
    prec = precision_score(y_test, y_pred_rf, zero_division=0)
    rec  = recall_score(y_test, y_pred_rf, zero_division=0)
    f1   = f1_score(y_test, y_pred_rf, zero_division=0)
    auc  = roc_auc_score(y_test, y_proba_rf)

    mlflow.log_params(RF_PARAMS)
    mlflow.log_metrics({
        "accuracy":  acc,
        "precision": prec,
        "recall":    rec,
        "f1_score":  f1,
        # 4th metric — AUC logged as well (log as many as needed; 4 minimum)
    })
    # Logging AUC separately so it's visible as its own metric row
    mlflow.log_metric("roc_auc", auc)

    # --- Log model ---
    mlflow.sklearn.log_model(rf, artifact_path="random_forest_model")

    # --- 2 Artifacts ---
    with tempfile.TemporaryDirectory() as tmpdir:
        cm_path = os.path.join(tmpdir, "rf_confusion_matrix.png")
        fi_path = os.path.join(tmpdir, "rf_feature_importance.png")
        save_confusion_matrix(y_test, y_pred_rf, "RF Confusion Matrix", cm_path)
        save_feature_importance(rf, FEATURES, fi_path)
        mlflow.log_artifact(cm_path)
        mlflow.log_artifact(fi_path)

    rf_run_id = rf_run.info.run_id
    print(f"✅ RF Run ID: {rf_run_id}")
    print(f"   Accuracy={acc:.3f}  Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}  AUC={auc:.3f}")

In [0]:
# Model 2 — Gradient Boosting Classifier
GB_PARAMS = {
    "n_estimators":    150,
    "learning_rate":   0.05,
    "max_depth":         4,
    "subsample":        0.8,
    "random_state":     42,
}

with mlflow.start_run(run_name="GradientBoosting_Podium") as gb_run:

    # --- Train ---
    gb = GradientBoostingClassifier(**GB_PARAMS)
    gb.fit(X_train, y_train)

    # --- Predict ---
    y_pred_gb  = gb.predict(X_test)
    y_proba_gb = gb.predict_proba(X_test)[:, 1]

    # --- 4 Metrics ---
    acc_g  = accuracy_score(y_test, y_pred_gb)
    prec_g = precision_score(y_test, y_pred_gb, zero_division=0)
    rec_g  = recall_score(y_test, y_pred_gb, zero_division=0)
    f1_g   = f1_score(y_test, y_pred_gb, zero_division=0)
    auc_g  = roc_auc_score(y_test, y_proba_gb)

    mlflow.log_params(GB_PARAMS)
    mlflow.log_metrics({
        "accuracy":  acc_g,
        "precision": prec_g,
        "recall":    rec_g,
        "f1_score":  f1_g,
    })
    mlflow.log_metric("roc_auc", auc_g)

    # --- Log model ---
    mlflow.sklearn.log_model(gb, artifact_path="gradient_boosting_model")

    # --- 2 Artifacts ---
    with tempfile.TemporaryDirectory() as tmpdir:
        cm_path = os.path.join(tmpdir, "gb_confusion_matrix.png")
        fi_path = os.path.join(tmpdir, "gb_feature_importance.png")
        save_confusion_matrix(y_test, y_pred_gb, "GB Confusion Matrix", cm_path)
        save_feature_importance(gb, FEATURES, fi_path)
        mlflow.log_artifact(cm_path)
        mlflow.log_artifact(fi_path)

    gb_run_id = gb_run.info.run_id
    print(f"✅ GB Run ID: {gb_run_id}")
    print(f"   Accuracy={acc_g:.3f}  Precision={prec_g:.3f}  Recall={rec_g:.3f}  F1={f1_g:.3f}  AUC={auc_g:.3f}")

In [0]:
# Write RF predictions → hw5_rf_predictions
from pyspark.sql.functions import lit, current_timestamp, col

rf_preds_pd = meta_test.copy()
rf_preds_pd['actual_podium']    = y_test.values
rf_preds_pd['predicted_podium'] = y_pred_rf
rf_preds_pd['predicted_proba']  = y_proba_rf
rf_preds_pd['model_name']       = "RandomForestClassifier"

rf_preds_spark = (spark.createDataFrame(rf_preds_pd)
    .withColumn("actual_podium", col("actual_podium").cast("int"))
    .withColumn("predicted_podium", col("predicted_podium").cast("int"))
    .withColumn("logged_at", current_timestamp()))

rf_preds_spark.write.mode("overwrite").saveAsTable(f"{YOUR_DB}.hw5_rf_predictions")
print(f"✅ RF predictions written to {YOUR_DB}.hw5_rf_predictions")
spark.sql(f"SELECT * FROM {YOUR_DB}.hw5_rf_predictions LIMIT 5").show()

In [0]:
# Write GB predictions → hw5_gb_predictions
gb_preds_pd = meta_test.copy()
gb_preds_pd['actual_podium']    = y_test.values
gb_preds_pd['predicted_podium'] = y_pred_gb
gb_preds_pd['predicted_proba']  = y_proba_gb
gb_preds_pd['model_name']       = "GradientBoostingClassifier"

gb_preds_spark = (spark.createDataFrame(gb_preds_pd)
    .withColumn("actual_podium", col("actual_podium").cast("int"))
    .withColumn("predicted_podium", col("predicted_podium").cast("int"))
    .withColumn("logged_at", current_timestamp()))

gb_preds_spark.write.mode("overwrite").saveAsTable(f"{YOUR_DB}.hw5_gb_predictions")
print(f"✅ GB predictions written to {YOUR_DB}.hw5_gb_predictions")
spark.sql(f"SELECT * FROM {YOUR_DB}.hw5_gb_predictions LIMIT 5").show()

In [0]:
# Side-by-side comparison
comparison = pd.DataFrame({
    "Model":     ["RandomForest", "GradientBoosting"],
    "Accuracy":  [acc,   acc_g],
    "Precision": [prec,  prec_g],
    "Recall":    [rec,   rec_g],
    "F1":        [f1,    f1_g],
    "ROC-AUC":   [auc,   auc_g],
    "MLflow Run ID": [rf_run_id, gb_run_id],
})
print(comparison.to_string(index=False))

In [0]:
# Verify tables exist and row counts
for tbl in ["hw5_rf_predictions", "hw5_gb_predictions"]:
    count = spark.sql(f"SELECT COUNT(*) as n FROM {YOUR_DB}.{tbl}").collect()[0]['n']
    print(f"{YOUR_DB}.{tbl}: {count} rows")